# MEIKA Vox · transcripción de audios desde Google Drive

Herramienta abierta para procesar entrevistas, talleres, reuniones y grupos focales por lotes.

1. En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, selecciona **GPU** si está disponible.
2. Pulsa **Entorno de ejecución → Ejecutar todas** y autoriza el acceso a **tu propio Drive**.
3. En el formulario escribe un nombre de proyecto, selecciona idioma y carpeta.
4. Presiona **Transcribir carpeta**.
5. Encontrarás los TXT y respaldos en **Mi unidad / MEIKA_Vox / Proyectos / nombre_del_proyecto / Transcripciones**.

La transcripción es preliminar y requiere revisión humana. Colab gratuito no garantiza GPU ni duración de sesión; mantén la sesión conectada durante el procesamiento. Separar hablantes requiere un token de Hugging Face y acceso autorizado al modelo pyannote.

**Privacidad:** los archivos no se publican en GitHub por utilizar este cuaderno. Revisa las condiciones de Google Colab y los modelos externos antes de procesar información sensible.


In [ ]:
import os
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path

REPO = Path("/content/meika-vox")


def checked(cmd, **kwargs):
    subprocess.run([str(item) for item in cmd], check=True, **kwargs)


checked(["apt-get", "-qq", "update"])
checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"])
if not (REPO / ".git").exists():
    checked(["git", "clone", "https://github.com/MeikaLab/meika-vox.git", REPO])
else:
    checked(["git", "-C", REPO, "pull", "--ff-only"])
# Colab ya trae un entorno Python funcional. No crear otro con venv:
# algunas imágenes de Colab no incorporan ensurepip para /usr/bin/python3.
PYTHON = Path(sys.executable)
print(f"Instalando MEIKA Vox para Python {sys.version.split()[0]} ...")
checked([PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "-e", str(REPO) + "[whisperx]"])
VOX = Path(sysconfig.get_path("scripts")) / "meika-vox"
if not VOX.is_file():
    executable = shutil.which("meika-vox")
    if executable:
        VOX = Path(executable)
if not VOX.is_file():
    raise RuntimeError("No se encontró meika-vox tras instalar. Revisa los mensajes anteriores.")
checked([VOX, "doctor", "--strict"])
print("Instalación comprobada. Ahora conecta Google Drive.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import importlib.util
import os
import re
import subprocess
import unicodedata
from collections import Counter
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

ROOT = Path("/content/drive/MyDrive").resolve()
OUTPUT_ROOT = ROOT / "MEIKA_Vox" / "Proyectos"
spec = importlib.util.spec_from_file_location("vox_batch", REPO / "scripts/colab_batch.py")
batch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(batch)

project_input = widgets.Text(
    value="Mi proyecto", description="Proyecto:",
    placeholder="Nombre de tu estudio",
    layout=widgets.Layout(width="95%")
)
language_input = widgets.Dropdown(
    options=[("Español", "es"), ("Inglés", "en"), ("Portugués", "pt"), ("Francés", "fr")],
    value="es", description="Idioma:"
)
glossary_input = widgets.Text(
    value="", description="Glosario:",
    placeholder="Opcional: archivo .json de tu Drive",
    layout=widgets.Layout(width="95%")
)
diarize_input = widgets.Checkbox(value=False, description="Separar hablantes (requiere HF_TOKEN)")
force_input = widgets.Checkbox(value=False, description="Volver a transcribir los ya terminados")

# Se incluyen carpetas superiores para permitir procesar subcarpetas por lotes.
print("Buscando carpetas con audio en tu Google Drive…")
audios = batch.discover_audio(ROOT, exclude=OUTPUT_ROOT)
counts = Counter(p.parent for p in audios)
for folder in tuple(counts):
    ancestor = folder.parent
    while ancestor.is_relative_to(ROOT) and ancestor != ROOT:
        counts.setdefault(ancestor, 0)
        ancestor = ancestor.parent
if not counts:
    raise RuntimeError("No hay audios compatibles en tu Drive. Sube una carpeta y reejecuta esta celda.")
choices = [
    (str(p.relative_to(ROOT)), str(p))
    for p in sorted(counts, key=lambda x: str(x).casefold())
]
folder_picker = widgets.Dropdown(options=choices, description="Carpeta:", layout=widgets.Layout(width="95%"))
status = widgets.HTML()
start_button = widgets.Button(
    description="Transcribir carpeta", button_style="success", layout=widgets.Layout(width="220px")
)
log = widgets.Output()


def selected_folder():
    folder = Path(folder_picker.value).resolve()
    if not folder.is_relative_to(ROOT):
        raise ValueError("Selecciona una carpeta dentro de tu Drive.")
    return folder


def update_count(change=None):
    folder = selected_folder()
    count = len(batch.discover_audio(folder, exclude=OUTPUT_ROOT))
    status.value = f"<b>{count} audios</b> en la carpeta y sus subcarpetas."


def project_slug(value):
    simplified = unicodedata.normalize("NFKD", value.strip())
    simplified = "".join(c for c in simplified if not unicodedata.combining(c))
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", simplified).strip("_")[:64]
    if not slug:
        raise ValueError("Escribe un nombre de proyecto válido.")
    return slug


folder_picker.observe(update_count, names="value")
update_count()


def start_batch(button):
    controls = [start_button, folder_picker, project_input,
                language_input, glossary_input, diarize_input, force_input]
    for control in controls:
        control.disabled = True
    log.clear_output()
    with log:
        try:
            import json

            project_id = project_slug(project_input.value)
            folder = selected_folder()
            output = OUTPUT_ROOT / project_id / "Transcripciones"
            glossary = None
            if glossary_input.value.strip():
                glossary = (ROOT / glossary_input.value.strip()).resolve()
                if not glossary.is_relative_to(ROOT) or not glossary.is_file():
                    raise ValueError("El glosario debe ser un archivo existente dentro de tu Drive.")
                if glossary.suffix.lower() != ".json":
                    raise ValueError("El glosario debe ser un archivo .json.")
            if diarize_input.value:
                from getpass import getpass

                try:
                    from google.colab import userdata
                    token = userdata.get("HF_TOKEN")
                except Exception:
                    token = None
                token = token or getpass("HF_TOKEN (oculto): ").strip()
                if not token:
                    raise RuntimeError("Falta HF_TOKEN: desmarca separación de hablantes o configúralo.")
                os.environ["HF_TOKEN"] = token
                print("Debes haber aceptado las condiciones del modelo pyannote.")
            runtime = json.loads(
                subprocess.check_output([str(VOX), "doctor", "--strict"], text=True)
            )
            device = "cuda" if runtime["cuda_available"] else "cpu"
            model = "large-v3" if device == "cuda" else "small"
            command = [
                str(VOX),
                "--project-id", project_id,
                "--language", language_input.value,
                "--model", model,
                "--device", device,
                "--batch-size", "4" if device == "cuda" else "1",
            ]
            if glossary is not None:
                command += ["--glossary", str(glossary)]
            if diarize_input.value:
                command.append("--diarize")
            revision = subprocess.check_output(
                ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
            ).strip()
            configuration = revision + (batch.fingerprint(glossary) if glossary else "")
            print(f"Proyecto: {project_id} | Motor: {model} | Origen: {folder}")
            print(f"Destino: {output}")
            print("Mantén Colab conectado mientras procesa.")
            report = batch.process_folder(
                folder, output, command,
                force=force_input.value,
                configuration=configuration
            )
            if report["failed"]:
                print("Hay audios pendientes: revisa los errores y vuelve a ejecutar.")
            else:
                print(f"Terminado. Revisa en tu Drive: {output.relative_to(ROOT)}")
        except Exception as exc:
            print("No se completó:", exc)
        finally:
            for control in controls:
                control.disabled = False


start_button.on_click(start_batch)
display(widgets.VBox([
    widgets.HTML("<h3>MEIKA Vox · Transcripción por lotes</h3>"),
    project_input, language_input, glossary_input,
    folder_picker, status, diarize_input, force_input, start_button, log,
]))
